In [0]:
%run ./00_config_setup


#### 1. S3 bucket path

#### All Utilities

In [0]:

spark.sql(f"USE CATALOG {CATALOG}")


_log_buffer = []   # collect all test results, flush once per domain

def log_test(domain, table_name, test_name, test_category,
             rows_checked, rows_failed,
             threshold_pct=None, notes=""):
    pass_rate = round(
        (rows_checked - rows_failed) / rows_checked * 100, 4
    ) if rows_checked > 0 else 100.0

    if threshold_pct is None:
        status = "PASS" if rows_failed == 0 else "FAIL"
    else:
        if pass_rate >= threshold_pct:
            status = "PASS"
        elif pass_rate >= max(threshold_pct - 5, 0):
            status = "WARN"
        else:
            status = "FAIL"

    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}[status]
    print(f"  {icon} [{test_category}] {test_name}"
          f"  →  failed={rows_failed:,} / {rows_checked:,}  [{status}]")

    _log_buffer.append((
        str(uuid.uuid4()), RUN_ID, RUN_TS, RUN_DT,
        domain, table_name, test_name, test_category,
        int(rows_checked), int(rows_failed),
        float(threshold_pct) if threshold_pct is not None else None,
        float(pass_rate),
        status,
        notes
    ))
    return status


def write_quarantine(df_bad: DataFrame, s3_path: str, table_name: str,
                     dq_col: str = "dq_fail_reasons"):
    bad_count = df_bad.count()
    if bad_count == 0:
        print(f"No bad rows")
        return 0

    (
        df_bad
        .withColumn("quarantine_run_id", lit(RUN_ID))
        .withColumn("quarantine_date",   lit(RUN_DT))
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(s3_path)
    )
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name}
        USING DELTA LOCATION '{s3_path}'
    """)
    return bad_count


def flush_logs():
    if not _log_buffer:
        return
    cols = [
        "log_id", "run_id", "run_timestamp", "run_date",
        "domain", "table_name", "test_name", "test_category",
        "rows_checked", "rows_failed", "threshold_pct",
        "pass_rate_pct", "status", "notes"
    ]
    (
        spark.createDataFrame(_log_buffer, schema=cols)
        .write.format("delta").mode("append")
        .save(S3_DQ_LOGS)
    )
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TBL_DQ_LOGS}
        USING DELTA LOCATION '{S3_DQ_LOGS}'
    """)
    _log_buffer.clear()

#### 2. unity catalog path

#### 3. run identity

#### DOMAIN 1: ELECTRONICS DQ

In [0]:
df_e = spark.read.format("delta").load(S3_SILVER_ELECTRONICS)

if spark.catalog.tableExists("retailflow360.dq.dq_checkpoint"):
    processed_batches = spark.table("retailflow360.dq.dq_checkpoint") \
        .filter(col("domain") == "electronics") \
        .select("batch_id")

    df_e = df_e.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )


e_tot = df_e.count()

if e_tot == 0:
    print("  No new data for DQ — skipping")
else:
    print(f"  Silver rows: {e_tot:,}")

    # Count each rule violation
    e_null_pk  = df_e.filter(col("order_id").isNull() | (trim(col("order_id")) == "")).count()
    e_bad_year = df_e.filter(col("order_year").isNotNull() & (col("order_year") != 2019)).count()
    e_bad_qty  = df_e.filter(col("quantity_ordered").isNull() | (col("quantity_ordered") <= 0)).count()
    e_bad_price= df_e.filter(col("price_each").isNull() | (col("price_each") <= 0)).count()
    e_bad_rev  = df_e.filter(col("line_revenue").isNull() | (col("line_revenue") <= 0)).count()

    e_rev_mismatch = df_e.filter(
        col("quantity_ordered").isNotNull() & col("price_each").isNotNull() &
        col("line_revenue").isNotNull() &
        (spark_abs(col("line_revenue") -
            spark_round(col("quantity_ordered").cast(DoubleType()) *
                        col("price_each"), 2)) > 0.02)
    ).count()
    e_no_state = df_e.filter(col("state").isNull() | (col("state") == "")).count()
    e_dup_ids  = df_e.filter(col("is_duplicate_order_id") == True).count()
    latest_e   = df_e.agg(spark_max("order_date_dt")).collect()[0][0]

    # Log all tests 
    log_test("electronics", TBL_SLV_ELECTRONICS,
            "order_id_not_null", "completeness", e_tot, e_null_pk,
            notes="PK must not be null")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "order_year_is_2019", "range", e_tot, e_bad_year,
            notes="All 12 monthly files are calendar year 2019")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "quantity_ordered_positive", "range", e_tot, e_bad_qty,
            notes="quantity_ordered must be > 0")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "price_each_positive", "range", e_tot, e_bad_price,
            notes="price_each must be > 0")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "line_revenue_positive", "accuracy", e_tot, e_bad_rev,
            notes="line_revenue must be > 0")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "line_revenue_calc_match", "accuracy", e_tot, e_rev_mismatch,
            threshold_pct=99.9,
            notes="line_revenue = qty * price ±$0.02 — soft check")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "state_code_populated", "completeness", e_tot, e_no_state,
            threshold_pct=99.5,
            notes="state must be parsed from purchase_address — soft check")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "order_id_uniqueness", "uniqueness", e_tot, e_dup_ids,
            threshold_pct=50.0,
            notes="Duplicate order_ids = multi-item orders — expected and acceptable")

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "latest_date_december_2019", "timeliness", 1,
            0 if (latest_e and str(latest_e) >= "2019-12-01") else 1,
            notes=f"Latest order date = {latest_e}  (expect Dec 2019)")


    # Bad = any hard rule violated
    df_e_bad = df_e.withColumn(
        "dq_fail_reasons",
        F.concat_ws(" | ",
            when(col("order_id").isNull() | (trim(col("order_id")) == ""),
                lit("null_order_id")),
            when(col("order_year").isNotNull() & (col("order_year") != 2019),
                lit("year_not_2019")),
            when(col("quantity_ordered").isNull() | (col("quantity_ordered") <= 0),
                lit("invalid_qty")),
            when(col("price_each").isNull() | (col("price_each") <= 0),
                lit("invalid_price")),
            when(col("line_revenue").isNull() | (col("line_revenue") <= 0),
                lit("invalid_revenue")),
        )
    ).filter(col("dq_fail_reasons") != "")

    e_quarantined = write_quarantine(df_e_bad, S3_Q_ELEC, TBL_Q_ELEC)

    log_test("electronics", TBL_SLV_ELECTRONICS,
            "overall_pass_rate", "consistency",
            e_tot, e_quarantined, threshold_pct=99.0,
            notes=f"Hard-rule violations quarantined: {e_quarantined:,}")

    flush_logs()

    df_e.select("batch_id").distinct() \
        .withColumn("domain", lit("electronics")) \
        .withColumn("processed_at", current_timestamp()) \
        .write.mode("append") \
        .saveAsTable("retailflow360.dq.dq_checkpoint")

    print(f"\n  Electronics DQ done  |  quarantined: {e_quarantined:,} rows")





  Silver rows: 178,437
  ✅ [completeness] order_id_not_null  →  failed=0 / 178,437  [PASS]
  ❌ [range] order_year_is_2019  →  failed=31 / 178,437  [FAIL]
  ✅ [range] quantity_ordered_positive  →  failed=0 / 178,437  [PASS]
  ✅ [range] price_each_positive  →  failed=0 / 178,437  [PASS]
  ✅ [accuracy] line_revenue_positive  →  failed=0 / 178,437  [PASS]
  ✅ [accuracy] line_revenue_calc_match  →  failed=0 / 178,437  [PASS]
  ✅ [completeness] state_code_populated  →  failed=0 / 178,437  [PASS]
  ✅ [uniqueness] order_id_uniqueness  →  failed=7,136 / 178,437  [PASS]
  ✅ [timeliness] latest_date_december_2019  →  failed=0 / 1  [PASS]
  ✅ [consistency] overall_pass_rate  →  failed=31 / 178,437  [PASS]

  Electronics DQ done  |  quarantined: 31 rows


#### DOMAIN 2: LIQUOR SALES DQ

In [0]:
df_l  = spark.read.format("delta").load(S3_SILVER_LIQUOR)

if spark.catalog.tableExists("retailflow360.dq.dq_checkpoint"):
    processed_batches = spark.table("retailflow360.dq.dq_checkpoint") \
        .filter(col("domain") == "liquor") \
        .select("batch_id")

    df_l = df_l.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )


l_tot = df_l.count()

if l_tot == 0:
    print("  No new data for DQ — skipping")
else:
    print(f"  Silver rows: {l_tot:,}")

    l_null_pk  = df_l.filter(
        col("invoice_item_number").isNull() | (trim(col("invoice_item_number")) == "")
    ).count()
    l_null_dt  = df_l.filter(col("transaction_date").isNull()).count()
    l_bad_year = df_l.filter(
        col("transaction_year").isNotNull() &
        ((col("transaction_year") < 2012) | (col("transaction_year") > 2020))
    ).count()

    # Soft
    l_zero_sale = df_l.filter(
        col("sale__dollars_").isNotNull() & (col("sale__dollars_") == 0)
    ).count()
    l_inv_margin = df_l.filter(
        col("state_bottle_retail").isNotNull() & col("state_bottle_cost").isNotNull() &
        (col("state_bottle_retail") < col("state_bottle_cost"))
    ).count()
    l_high_vol = df_l.filter(
        col("bottles_sold").isNotNull() & (col("bottles_sold") > 10000)
    ).count()
    l_dup_inv  = int(
        df_l.groupBy("invoice_item_number").count()
        .filter(col("count") > 1).agg(spark_sum("count"))
        .collect()[0][0] or 0
    )
    latest_l   = df_l.agg(spark_max("transaction_date")).collect()[0][0]
    earliest_l = df_l.agg(spark_min("transaction_date")).collect()[0][0]

    log_test("liquor", TBL_SLV_LIQUOR,
            "invoice_not_null", "completeness", l_tot, l_null_pk,
            notes="PK must not be null")

    log_test("liquor", TBL_SLV_LIQUOR,
            "transaction_date_not_null", "completeness", l_tot, l_null_dt,
            notes="Date required for all temporal analysis")

    log_test("liquor", TBL_SLV_LIQUOR,
            "transaction_year_2012_to_2020", "range", l_tot, l_bad_year,
            notes="Iowa liquor dataset covers 2012-2020 only")

    log_test("liquor", TBL_SLV_LIQUOR,
            "zero_sale_dollars", "accuracy", l_tot, l_zero_sale,
            threshold_pct=99.0,
            notes="Zero-dollar sales = credit/reversal transactions — known in Iowa data")

    log_test("liquor", TBL_SLV_LIQUOR,
            "retail_price_gte_cost", "accuracy", l_tot, l_inv_margin,
            threshold_pct=99.9,
            notes="retail < cost = inverted margin — 41 known rows in source")

    log_test("liquor", TBL_SLV_LIQUOR,
            "bottles_sold_max_10000", "range", l_tot, l_high_vol,
            threshold_pct=99.9,
            notes="Bulk orders > 10,000 bottles — flagged FLAGGED_HIGH_VOLUME in Silver")

    log_test("liquor", TBL_SLV_LIQUOR,
            "invoice_unique", "uniqueness", l_tot, l_dup_inv,
            notes="invoice_item_number must be unique — 0 expected")

    log_test("liquor", TBL_SLV_LIQUOR,
            "date_range_2012_2020", "timeliness", 1,
            0 if (latest_l and str(latest_l) <= "2020-12-31") else 1,
            notes=f"Date range: {earliest_l} → {latest_l}  (expect 2012-2020)")

    df_l_bad = (
        df_l
        .filter(col("row_quality_flag") != "CLEAN")
        .withColumn("dq_fail_reasons", col("row_quality_flag"))
        )

    l_quarantined = write_quarantine(df_l_bad, S3_Q_LIQ, TBL_Q_LIQ)

    log_test("liquor", TBL_SLV_LIQUOR,
            "overall_pass_rate", "consistency",
            l_tot, l_quarantined, threshold_pct=99.9,
            notes=f"Hard-rule violations quarantined: {l_quarantined:,}")

    flush_logs()

    df_l.select("batch_id").distinct() \
        .withColumn("domain", lit("liquor")) \
        .withColumn("processed_at", current_timestamp()) \
        .write.mode("append") \
        .saveAsTable("retailflow360.dq.dq_checkpoint")


    print(f"\n  Liquor DQ done  |  quarantined: {l_quarantined:,} rows")


  Silver rows: 19,666,763
  ✅ [completeness] invoice_not_null  →  failed=0 / 19,666,763  [PASS]
  ✅ [completeness] transaction_date_not_null  →  failed=0 / 19,666,763  [PASS]
  ✅ [range] transaction_year_2012_to_2020  →  failed=0 / 19,666,763  [PASS]
  ✅ [accuracy] zero_sale_dollars  →  failed=4,943 / 19,666,763  [PASS]
  ✅ [accuracy] retail_price_gte_cost  →  failed=41 / 19,666,763  [PASS]
  ✅ [range] bottles_sold_max_10000  →  failed=4 / 19,666,763  [PASS]
  ✅ [uniqueness] invoice_unique  →  failed=0 / 19,666,763  [PASS]
  ✅ [timeliness] date_range_2012_2020  →  failed=0 / 1  [PASS]
  ✅ [consistency] overall_pass_rate  →  failed=5,006 / 19,666,763  [PASS]

  Liquor DQ done  |  quarantined: 5,006 rows


#### DOMAIN 3: BOOKS DATA DQ

In [0]:
df_bd  = spark.read.format("delta").load(S3_SILVER_BOOKS_DATA)

if spark.catalog.tableExists("retailflow360.dq.dq_checkpoint"):
    processed_batches = spark.table("retailflow360.dq.dq_checkpoint") \
        .filter(col("domain") == "books_data") \
        .select("batch_id")

    df_bd = df_bd.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )


bd_tot = df_bd.count()

if bd_tot == 0:
    print("  No new data for DQ — skipping")
else:
    print(f"  Silver rows: {bd_tot:,}")

    bd_null_pk  = df_bd.filter(col("title").isNull() | (trim(col("title")) == "")).count()
    bd_dup_title = int(
        df_bd.groupBy("title").count()
        .filter(col("count") > 1).agg(spark_sum("count"))
        .collect()[0][0] or 0
    )
    bd_null_auth = df_bd.filter(col("primary_author").isNull()).count()
    bd_null_desc = df_bd.filter(col("description").isNull()).count()
    bd_future_yr = df_bd.filter(
        col("published_year").isNotNull() & (col("published_year") > 2025)
    ).count()

    log_test("books_data", TBL_SLV_BOOKS_DATA,
            "title_not_null", "completeness", bd_tot, bd_null_pk,
            notes="title is PK — required for Gold DIM_PRODUCT join")

    log_test("books_data", TBL_SLV_BOOKS_DATA,
            "title_unique", "uniqueness", bd_tot, bd_dup_title,
            notes="Each book should appear once in catalog")

    log_test("books_data", TBL_SLV_BOOKS_DATA,
            "primary_author_present", "completeness", bd_tot, bd_null_auth,
            threshold_pct=85.0,
            notes="~14.8% null authors in source — threshold 85%")

    log_test("books_data", TBL_SLV_BOOKS_DATA,
            "description_present", "completeness", bd_tot, bd_null_desc,
            threshold_pct=65.0,
            notes="~32% null descriptions in source — threshold 65%")

    log_test("books_data", TBL_SLV_BOOKS_DATA,
            "published_year_not_future", "range", bd_tot, bd_future_yr,
            threshold_pct=99.9,
            notes="Publication year should not be in the future")

    df_bd_bad = df_bd.withColumn(
        "dq_fail_reasons",
        F.concat_ws(" | ",
            when(col("title").isNull() | (trim(col("title")) == ""),
                lit("null_title")),
        )
    ).filter(col("dq_fail_reasons") != "")


    df_bd_bad = (
        df_bd
        .filter(~col("row_quality_flag").isin("CLEAN", "FLAGGED_NULL_DESC"))
        .withColumn("dq_fail_reasons", col("row_quality_flag"))
    )

    bd_quarantined = write_quarantine(df_bd_bad, S3_Q_BD, TBL_Q_BD)

    log_test("books_data", TBL_SLV_BOOKS_DATA,
            "overall_pass_rate", "consistency",
            bd_tot, bd_quarantined, threshold_pct=99.9,
            notes=f"Hard-rule violations quarantined: {bd_quarantined:,}")

    flush_logs()


    df_bd.select("batch_id").distinct() \
        .withColumn("domain", lit("books_data")) \
        .withColumn("processed_at", current_timestamp()) \
        .write.mode("append") \
        .saveAsTable("retailflow360.dq.dq_checkpoint")


    print(f"\n  Books Data DQ done  |  quarantined: {bd_quarantined:,} rows")



  Silver rows: 212,404
  ❌ [completeness] title_not_null  →  failed=1 / 212,404  [FAIL]
  ✅ [uniqueness] title_unique  →  failed=0 / 212,404  [PASS]
  ✅ [completeness] primary_author_present  →  failed=31,421 / 212,404  [PASS]
  ✅ [completeness] description_present  →  failed=68,442 / 212,404  [PASS]
  ✅ [range] published_year_not_future  →  failed=1 / 212,404  [PASS]
  ✅ [consistency] overall_pass_rate  →  failed=199 / 212,404  [PASS]

  Books Data DQ done  |  quarantined: 199 rows


#### DOMAIN 4: BOOKS RATING DQ|

In [0]:
df_br  = spark.read.format("delta").load(S3_SILVER_BOOKS_RATING)

if spark.catalog.tableExists("retailflow360.dq.dq_checkpoint"):
    processed_batches = spark.table("retailflow360.dq.dq_checkpoint") \
        .filter(col("domain") == "books_rating") \
        .select("batch_id")

    df_br = df_br.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )


br_tot = df_br.count()

if br_tot == 0:
    print("  No new data for DQ — skipping")
else:
    print(f"  Silver rows: {br_tot:,}")

    br_null_rid   = df_br.filter(col("review_id").isNull() | (trim(col("review_id")) == "")).count()
    br_null_bid   = df_br.filter(col("book_id").isNull() | (trim(col("book_id")) == "")).count()
    br_bad_score  = df_br.filter(
        col("review_score").isNull() |
        (col("review_score") < 1.0) | (col("review_score") > 5.0)
    ).count()
    br_bad_date   = df_br.filter(col("is_valid_review_date") == False).count()

    # Soft
    br_dup_rid    = int(
        df_br.groupBy("review_id").count()
        .filter(col("count") > 1).agg(spark_sum("count"))
        .collect()[0][0] or 0
    )
    br_anon       = df_br.filter(col("user_id") == "ANONYMOUS").count()
    br_help_err   = df_br.filter(
        col("helpful_votes").isNotNull() & col("total_votes").isNotNull() &
        (col("helpful_votes") > col("total_votes"))
    ).count()
    latest_br     = df_br.agg(spark_max("review_date")).collect()[0][0]
    earliest_br   = df_br.agg(spark_min("review_date")).collect()[0][0]

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "review_id_not_null", "completeness", br_tot, br_null_rid,
            notes="Surrogate PK must not be null")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "book_id_not_null", "completeness", br_tot, br_null_bid,
            notes="Amazon ASIN required for Gold product join")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "review_score_1_to_5", "range", br_tot, br_bad_score,
            notes="Amazon scores are 1.0-5.0 only")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "review_date_1995_to_2023", "range", br_tot, br_bad_date,
            notes="Amazon founded 1995 — pre-1995 = corrupt epoch")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "review_id_unique", "uniqueness", br_tot, br_dup_rid,
            notes="md5 surrogate review_id must be unique — 0 expected")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "anonymous_user_rate", "completeness", br_tot, br_anon,
            threshold_pct=75.0,
            notes=f"~18.7% ANONYMOUS expected. "
                f"Non-anonymous: {round((br_tot-br_anon)/br_tot*100,1)}%")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "helpful_votes_not_exceed_total", "accuracy", br_tot, br_help_err,
            notes="helpful_votes must not exceed total_votes — already capped in Silver")

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "review_date_range_check", "timeliness", br_tot, 0,
            notes=f"Review dates: {earliest_br} → {latest_br}")

    df_br_bad = df_br.withColumn(
        "dq_fail_reasons",
        F.concat_ws(" | ",
            when(col("review_id").isNull() | (trim(col("review_id")) == ""),
                lit("null_review_id")),
            when(col("book_id").isNull() | (trim(col("book_id")) == ""),
                lit("null_book_id")),
            when(
                col("review_score").isNull() |
                (col("review_score") < 1.0) | (col("review_score") > 5.0),
                lit("invalid_score")),
            when(col("is_valid_review_date") == False,
                lit("invalid_date")),
        )
    ).filter(col("dq_fail_reasons") != "")

    br_quarantined = write_quarantine(df_br_bad, S3_Q_BR, TBL_Q_BR)

    log_test("books_rating", TBL_SLV_BOOKS_RATING,
            "overall_pass_rate", "consistency",
            br_tot, br_quarantined, threshold_pct=99.0,
            notes=f"Hard-rule violations quarantined: {br_quarantined:,}")

    flush_logs()

    df_br.select("batch_id").distinct() \
        .withColumn("domain", lit("books_rating")) \
        .withColumn("processed_at", current_timestamp()) \
        .write.mode("append") \
        .saveAsTable("retailflow360.dq.dq_checkpoint")


    print(f"\n  Books Rating DQ done  |  quarantined: {br_quarantined:,} rows")


  Silver rows: 2,988,547
  ✅ [completeness] review_id_not_null  →  failed=0 / 2,988,547  [PASS]
  ✅ [completeness] book_id_not_null  →  failed=0 / 2,988,547  [PASS]
  ✅ [range] review_score_1_to_5  →  failed=0 / 2,988,547  [PASS]
  ✅ [range] review_date_1995_to_2023  →  failed=0 / 2,988,547  [PASS]
  ✅ [uniqueness] review_id_unique  →  failed=0 / 2,988,547  [PASS]
  ✅ [completeness] anonymous_user_rate  →  failed=556,878 / 2,988,547  [PASS]
  ✅ [accuracy] helpful_votes_not_exceed_total  →  failed=0 / 2,988,547  [PASS]
  ✅ [timeliness] review_date_range_check  →  failed=0 / 2,988,547  [PASS]
No bad rows
  ✅ [consistency] overall_pass_rate  →  failed=0 / 2,988,547  [PASS]

  Books Rating DQ done  |  quarantined: 0 rows
